# Fase 1 — Adquisición de datos (FotMob y StatsBomb)

Dos formas distintas de conseguir datos:

1. **FotMob**: no tiene API pública oficial, pero su web llama internamente a
   endpoints JSON (`https://www.fotmob.com/api/...`). Vamos a inspeccionar uno
   con `requests` — esto es "scraping" en el sentido de leer un JSON no
   documentado, no de parsear HTML.
2. **StatsBomb Open Data**: dataset abierto y documentado, con datos de
   eventos muy detallados (cada pase, tiro, regate, etc. con coordenadas).

In [22]:
import requests
import pandas as pd

pd.set_option("display.max_columns", 50)

## 1. FotMob — explorar un endpoint JSON

Antes de automatizar nada: abre https://www.fotmob.com en el navegador, entra
a un partido, abre las DevTools (pestaña Network, filtro Fetch/XHR) y busca
la llamada que trae los datos del partido. Copia esa URL aquí abajo.

Esto es intencional — cada curso enseña una URL de ejemplo distinta y suelen
cambiar; lo importante es el método (inspeccionar Network, no adivinar la URL).

In [23]:
headers = {"User-Agent": "Mozilla/5.0"}

# Reemplaza por la URL que encuentres en DevTools > Network
url_ejemplo = "https://www.fotmob.com/api/matchDetails?matchId=REEMPLAZA_ID"

# resp = requests.get(url_ejemplo, headers=headers)
# resp.status_code, resp.json()

## 2. StatsBomb Open Data

Flujo típico: `competitions()` -> `matches(competition_id, season_id)` ->
`events(match_id)`.

In [24]:
from statsbombpy import sb

competitions = sb.competitions()
competitions[["competition_id", "season_id", "competition_name", "season_name"]].head(15)

,competition_id,season_id,competition_name,season_name
0,9,281,1. Bundesliga,2023/2024
1,9,27,1. Bundesliga,2015/2016
2,1267,107,African Cup of Nations,2023
3,16,4,Champions League,2018/2019
4,16,1,Champions League,2017/2018
5,16,2,Champions League,2016/2017
6,16,27,Champions League,2015/2016
7,16,26,Champions League,2014/2015
8,16,25,Champions League,2013/2014
9,16,24,Champions League,2012/2013


In [25]:
# Elige una competition_id / season_id de la tabla anterior
# 43 / 3 = FIFA World Cup 2018, dataset abierto siempre disponible
competition_id = 43
season_id = 3

matches = sb.matches(competition_id=competition_id, season_id=season_id)
matches[["match_id", "home_team", "away_team", "match_date"]].head(10)

/Users/usuario/miniconda3/envs/futbol-bigdata/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,home_team,away_team,match_date
0,8650,Brazil,Belgium,2018-07-06
1,7584,Belgium,Japan,2018-07-02
2,7554,England,Panama,2018-06-24
3,7539,Poland,Senegal,2018-06-19
4,7550,Serbia,Switzerland,2018-06-22
5,7538,Sweden,South Korea,2018-06-18
6,7534,Germany,Mexico,2018-06-17
7,7543,Iran,Spain,2018-06-20
8,7544,Uruguay,Saudi Arabia,2018-06-20
9,7546,France,Peru,2018-06-21


In [26]:
match_id = 8650  # elige uno de match_id de la tabla anterior

events = sb.events(match_id=match_id)
events.shape

/Users/usuario/miniconda3/envs/futbol-bigdata/lib/python3.11/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


(3589, 80)

In [27]:
events["type"].value_counts()

type
Pass                 917
Ball Receipt*        872
Carry                757
Pressure             498
Ball Recovery         84
Duel                  44
Clearance             43
Block                 39
Foul Committed        38
Camera On             38
Foul Won              37
Goal Keeper           35
Shot                  33
Dribble               29
Miscontrol            28
Dispossessed          25
Dribbled Past         25
Interception          12
Camera off             7
Substitution           5
Half Start             4
Half End               4
Tactical Shift         3
Starting XI            2
Player Off             2
Player On              2
Referee Ball-Drop      2
Error                  1
Own Goal For           1
Own Goal Against       1
Injury Stoppage        1
Name: count, dtype: int64

## Guardar para las siguientes fases

Guardamos los eventos de este partido como parquet en `data/processed/` para
no tener que volver a descargarlos en los notebooks de visualización.

In [28]:
output_path = "../data/processed/events_ejemplo.pkl"
events.to_pickle(output_path)
print(f"Guardado en {output_path}")

Guardado en ../data/processed/events_ejemplo.pkl


## Conceptos clave de esta fase

- **Event data** (StatsBomb): una fila por acción (pase, tiro, regate...) con
  coordenadas `x, y` de inicio y a veces de fin. Es lo que usaremos para shot
  maps, pass networks y heat maps.
- **Tracking data**: posición de los 22 jugadores + balón cada ~0.1s. No está
  en el dataset abierto de StatsBomb (es un producto de pago), pero es la base
  de métricas físicas (distancia recorrida, sprints, velocidad).
- El sistema de coordenadas de StatsBomb es 120x80 (largo x ancho de cancha).

Siguiente notebook: `02_visualizacion_equipo.ipynb`.